# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides an end-to-end guide for loading, exploring, and processing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://mlcommons.org/croissant/) library. The notebook follows best practices for referencing entities via their `@id` and demonstrates dynamic, programmatic dataset management and EDA workflows.

### Dataset Source
The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` and auxiliary libraries are installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading

Load the dataset metadata and record sets using `mlcroissant`. We'll print the dataset overview as described in the Croissant metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object (don't iterate or subscript as dict)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview

In this section, we review available record sets, their `@id`s, all field `@id`s within each record set, and provide short descriptions where available. All entity references (**record sets**, **fields**) use their unique `@id` as required.

**Note:** All entity lists are dynamically discovered from the loaded metadata.

In [ ]:
# Explore available record sets and their fields
print("Available Record Sets and Fields:\n")

all_record_sets = list(dataset.metadata.record_sets)
recordsets_dict = {}
for rec in all_record_sets:
    print(f"- RecordSet: id={rec.id!r}, name='{rec.name}', description='{getattr(rec, 'description', '-')}'")
    fields = list(rec.fields)
    field_info = []
    for field in fields:
        # Show field id, name, datatype
        field_info.append({
            'id': field.id, 'name': field.name, 'dataType': getattr(field, 'data_type', '-')
            })
        print(f"    * Field: id={field.id!r}, name='{field.name}', dataType={getattr(field, 'data_type', '-')}")
    recordsets_dict[rec.id] = field_info

if not all_record_sets:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction

This section demonstrates how to extract tabular data from a record set:
 - **Choose** one or more record sets by their `@id` (see above for IDs).
 - Data is loaded into a pandas DataFrame using `dataset.records(record_set=<@id>)`.

_For this dataset, we'll load each discovered record set._

In [ ]:
# Gather all record set `@id`s (should match those seen above)
record_set_ids = [rs.id for rs in all_record_sets]
print("RecordSet IDs:", record_set_ids)

dataframes = dict()
for rec_id in record_set_ids:
    print(f"\nLoading records for record set: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    # If records empty, skip
    if not records:
        print("  (No records found)")
        continue
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"  Loaded {len(df)} rows, columns: {df.columns.tolist()}")
    # Show first few rows
    display(df.head())

# For demonstration, pick the main record set (first one by default)
selected_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)

We now apply basic EDA operations to the data, using only column names that correspond to the `@id` field.

**Steps:**
 - Select a numeric field by its field `@id`
 - Filter records on a threshold
 - Normalize the numeric field
 - Group by a categorical field

These operations are done using the main record set and its fields discovered earlier.

In [ ]:
# Identify a numeric field and a group field (by @id) from the record set
main_record_set_id = selected_record_set_id
if main_record_set_id is None or main_record_set_id not in dataframes:
    print("No tabular record set found, cannot proceed with EDA.")
else:
    main_df = dataframes[main_record_set_id].copy()

    # List available columns (these correspond to field @id)
    print("Available columns (field @id):", main_df.columns.tolist())

    # Attempt to pick a numeric field (e.g. score, age, etc.) by data type or name
    # For demo, try to pick first float/integer column if available
    numeric_field_id = None
    for field in recordsets_dict[main_record_set_id]:
        if field['dataType'] in ('Float', 'Integer', 'Number') and field['id'] in main_df.columns:
            numeric_field_id = field['id']
            break
    # If not found, just pick the first possible column
    if not numeric_field_id and main_df.shape[1] > 0:
        numeric_field_id = main_df.columns[0]

    print(f"Selected numeric field '@id': {numeric_field_id}")

    threshold = main_df[numeric_field_id].quantile(0.75) if numeric_field_id else 10

    # Filter
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered {len(filtered_df)}/{len(main_df)} records with {numeric_field_id} > {threshold:.3f}.")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Example normalization of {numeric_field_id} (first 5 rows):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by first non-numeric field
    group_field_id = None
    for field in recordsets_dict[main_record_set_id]:
        if field['dataType'] not in ('Float', 'Integer', 'Number') and field['id'] in main_df.columns:
            group_field_id = field['id']
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by field {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical field for grouping found.")

## 5. Visualization

Visualize numeric field distributions and possible relationships using matplotlib. Field names are always referenced by their `@id`.

**Example:** Histogram of the selected numeric field by group, if grouping is possible.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id is not None and numeric_field_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Histogram for the numeric field
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].dropna().hist(bins=20, edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If a grouping field was determined earlier, plot group averages
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        grouped_df.set_index(group_field_id)[numeric_field_id].plot(kind='bar')
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No record set or numeric field available for visualization.")

## 6. Conclusion

- This notebook demonstrated how to load and explore a dataset described with the Croissant schema using `mlcroissant` and reference all fields, record sets, and data elements using their `@id`.
- We performed basic EDA including filtering, normalization, grouping, and visualization on tabular data extracted from the metadata and record sets.
- All steps are reproducible and can be adapted to other [Croissant](https://mlcommons.org/croissant/) datasets with appropriate updates to the schema URL and entity `@id`s.

For details and advanced operations, consult the [`mlcroissant` documentation](https://github.com/mlcommons/croissant).